In [ ]:
import pandas as pd

# 读取原始数据
df = pd.read_excel('10minwinddata.xlsx')

# 确保时间列是datetime类型
df['Time'] = pd.to_datetime(df['Time'])

# 创建1分钟间隔的时间序列
start_time = df['Time'].min()
end_time = df['Time'].max()
time_1min = pd.date_range(start=start_time, end=end_time, freq='1min')

# 线性插值（所有站点同时插值）
df_indexed = df.set_index('Time')
df_1min_indexed = df_indexed.reindex(time_1min)
df_1min_indexed[['Site1', 'Site2', 'Site3']] = df_indexed[['Site1', 'Site2', 'Site3']].reindex(time_1min).interpolate(method='linear')

# 重置索引并保存为CSV（无行数限制）
df_1min = df_1min_indexed.reset_index().rename(columns={'index': 'Time'})
df_1min.to_csv('1min_wind_data_linear_interpolated.csv', index=False)

print(f"插值完成！共生成 {len(df_1min):,} 条1分钟数据，已保存为CSV。")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
# 设置中文显示
plt.rcParams["font.family"] = ["Heiti TC"]
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# -------------------------- 1. 读取数据 --------------------------
# 原始10分钟数据（可选，用于统计特性）
df_original = pd.read_excel('10minwinddata.xlsx')
# 线性插值后的1分钟数据（需先运行之前的插值代码）
df_interpolated = pd.read_csv('1min_wind_data_linear_interpolated.csv')

# 转换时间格式
df_original['Time'] = pd.to_datetime(df_original['Time'])
df_interpolated['Time'] = pd.to_datetime(df_interpolated['Time'])

# -------------------------- 2. 配置波动参数 --------------------------
sites = ['Site1', 'Site2', 'Site3']
noise_params = {
    'white_noise_std': 0.1,       # 高频白噪声标准差（占原始风速的比例，如0.2表示20%波动）
    'diurnal_amplitude': 0.1,     # 日周期波动幅度（原始风速的10%）
    'clip_min': 0                 # 风速最小值（避免负数）
}

# -------------------------- 3. 生成复合随机波动 --------------------------
def generate_wind_noise(df, sites, params):
    """
    生成符合风速特性的复合随机波动：
    - 高频白噪声（短时间尺度随机起伏）
    - 低频日周期波动（缓慢的日变化）
    """
    n = len(df)
    t = np.arange(n)  # 时间索引（分钟）
    
    # 1. 高频白噪声（每个站点独立生成）
    white_noise = np.zeros((n, len(sites)))
    for i, site in enumerate(sites):
        # 白噪声标准差：基于原始数据的标准差调整
        site_std = df_original[site].std()
        white_noise[:, i] = norm.rvs(
            loc=0,
            scale=site_std * params['white_noise_std'],
            size=n
        )
    
    # 2. 低频日周期波动（所有站点共享日趋势）
    diurnal_cycle = np.sin(2 * np.pi * t / 1440)  # 1440分钟=1天，周期为1天
    diurnal_noise = diurnal_cycle.reshape(-1, 1) * params['diurnal_amplitude']
    
    # 3. 合成总波动（高频+低频）
    total_noise = white_noise + diurnal_noise

    for i in range(len(total_noise)):
        if i % 10 == 0:
            total_noise[i]=0
        else:
            None
    return total_noise

# 生成波动
noise = generate_wind_noise(df_interpolated, sites, noise_params)

# -------------------------- 4. 叠加波动并修正数据 --------------------------
df_noisy = df_interpolated.copy()
for i, site in enumerate(sites):
    # 叠加波动
    df_noisy[site] = df_interpolated[site] + noise[:, i]

# -------------------------- 5. 结果验证与保存 --------------------------
# 统计修正后的数据特性
print("=== 插值+波动后数据统计 ===")
print(f"数据量: {len(df_noisy)} 条（1分钟间隔）")
print(f"Site1 波动后均值: {df_noisy['Site1'].mean():.2f} | 原始插值均值: {df_interpolated['Site1'].mean():.2f}")
print(f"Site2 波动后均值: {df_noisy['Site2'].mean():.2f} | 原始插值均值: {df_interpolated['Site2'].mean():.2f}")
print(f"Site3 波动后均值: {df_noisy['Site3'].mean():.2f} | 原始插值均值: {df_interpolated['Site3'].mean():.2f}")

# 保存带波动的结果
df_noisy.to_csv('1min_wind_data_with_noise.csv', index=False)
print("带随机波动的结果已保存到 '1min_wind_data_with_noise.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

try:
    # 读取CSV文件
    df = df_interpolated.copy()
    
    # 打印前5行，检查数据格式
    print("数据前5行：")
    print(df.head())
    
    # 将第一列转换为时间序列
    df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
    
    # 检查时间转换是否成功
    if df.iloc[:, 0].isna().any():
        print("警告：部分时间数据无法解析，可能格式不一致！")
    
    # 提取时间和数据
    time = df.iloc[:, 0]
    data = pd.to_numeric(df.iloc[:, 3], errors='coerce')  # 确保第二列是数值
    
    # 检查数据是否有效
    if data.isna().any():
        print("警告：数据列中存在无效值（非数值），已转换为NaN！")
    
    # 计算布林通道（周期=100，k=2）
    window = 500
    k = 1
    # 中轨：100周期移动平均
    middle_band = data.rolling(window=window).mean()
    # 标准差
    std = data.rolling(window=window).std()
    # 上轨和下轨
    upper_band = middle_band + k * std
    lower_band = middle_band - k * std
    
    # 绘制折线图和布林通道
    plt.figure(figsize=(12, 6))  # 增大画布，适应大数据
    # 绘制原始数据
    plt.plot(time, data, color='blue', label='Wind Data', linewidth=1)
    # 绘制布林通道
    plt.plot(time, middle_band, color='orange', label='Middle Band (SMA 100)', linewidth=1)
    plt.plot(time, upper_band, color='green', label='Upper Band', linewidth=1)
    plt.plot(time, lower_band, color='red', label='Lower Band', linewidth=1)
    # 填充上轨和下轨之间的区域
    plt.fill_between(time, lower_band, upper_band, color='gray', alpha=0.2, label='Bollinger Band')
    
    # 设置Y轴标签和标题
    plt.ylabel('Data Value')
    plt.title('1-Minute Wind Data with Bollinger Bands (Period=100)')
    plt.legend()
    plt.grid(True)
    
    # 移除X轴标签和刻度
    plt.gca().set_xticks([])  # 不显示X轴刻度
    plt.gca().set_xlabel('')  # 移除X轴标题
    
    # 优化布局
    plt.tight_layout()
    
    # 保存图片（防止显示卡顿）
    plt.savefig('wind_plot_with_bollinger.png', dpi=100)
    print("图表已保存为 wind_plot_with_bollinger.png，请检查当前目录！")
    
    # 显示图形（如果环境支持）
    plt.show()

except Exception as e:
    print(f"出错啦：{e}")

In [ ]:
# 统计数值大于上轨或小于下轨的点
outliers = data[(data > upper_band) | (data < lower_band)]
outlier_count = len(outliers)
outlier_percentage = (outlier_count / len(data)) * 100
divisible_by_10 = outliers[outliers.index % 10 == 0]
index_list = divisible_by_10.index / 10
df_site3 = pd.read_excel('10minwinddata.xlsx')
df_site3['index_flag'] = df_site3.index.isin(index_list).astype(int)
df_site3.to_excel('df_site3.xlsx')

In [ ]:
site = 'Site3'
df = pd.read_excel('df_'+site+'.xlsx')
# 筛选出index_flag为1的数据点
flagged_points = df[df['index_flag'] == 1]
flagged_pointss = df[df['index_flag'] == 0]

# 绘制图表
plt.figure(figsize=(12, 6))

# 绘制Site1随时间的折线图
plt.plot(df['Time'], df[site], label=site, color=plt.cm.tab20c(0), linewidth=1.5)

# 标注index_flag为1的点
plt.scatter(flagged_points['Time'], flagged_points[site], 
           color='red', s=50, label='Index Flag = 1', zorder=5)
plt.scatter(flagged_pointss['Time'], flagged_pointss[site], 
           color='green', s=50, label='Index Flag = 1', zorder=5)
# 添加标题和标签
plt.title(site+'随时间变化折线图（标注index_flag为1的点）')
plt.xlabel('时间')
plt.ylabel(site+'值')
plt.legend()
plt.grid(True, alpha=0.3)

# 优化时间轴显示
plt.xticks(rotation=45)
plt.tight_layout()

# 保存图表
img_path = site+'_time_series_with_flags.png'
plt.savefig(img_path)
print(f"图表已保存为 {img_path}")

In [ ]:
# -*- coding: utf-8 -*-
"""
Aventa AV-7 SCADA:
1) Build regime labels from high-frequency wind speed using:
   - WMO-style 3s gust metric
   - TI (turbulence intensity)
   - Gust factor
   - IEC-inspired turbulence proxy (engineering proxy, not exact IEC certification formula)
2) Run Bollinger-band financial strategy
3) Compare predicted labels vs. standard labels
4) Produce paper-ready figures

Author: you
"""

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.dates import DateFormatter
from matplotlib.colors import ListedColormap
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score, recall_score,
    f1_score, accuracy_score, balanced_accuracy_score, cohen_kappa_score
)

# =========================================================
# 0. Global plotting style
# =========================================================
plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


# =========================================================
# 1. Load CSV + detect time column / wind speed column
# =========================================================
CSV_PATH = "Aventa_AV7_IET_OST_SCADA.csv"   # 改成你的实际路径
OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- helper: detect time column ----------
def detect_time_col(df: pd.DataFrame):
    candidates = []
    for c in df.columns:
        cl = str(c).lower()
        if any(k in cl for k in ["time", "date", "datetime", "timestamp"]):
            candidates.append(c)

    if len(candidates) == 0:
        # fallback: try first few columns
        candidates = list(df.columns[:5])

    best_col = None
    best_valid = -1
    for c in candidates:
        parsed = pd.to_datetime(df[c], errors="coerce")
        valid = parsed.notna().sum()
        if valid > best_valid:
            best_valid = valid
            best_col = c

    if best_col is None:
        raise ValueError("Cannot detect a valid time column.")
    return best_col


# ---------- helper: detect wind speed column ----------
def detect_windspeed_col(df: pd.DataFrame):
    # prioritize columns with wind + speed semantics
    priority_patterns = [
        "windspeed", "wind_speed", "wind speed", "windspeedmean",
        "windspeedavg", "avgwindspeed", "anemometer", "nacellewindspeed",
        "ambientwindspeed", "windspeed1", "windspeed2"
    ]

    cols = list(df.columns)
    lower_map = {c: str(c).lower().replace("_", "").replace(" ", "") for c in cols}

    # exact/pattern priority
    for p in priority_patterns:
        p2 = p.replace("_", "").replace(" ", "")
        for c, cl in lower_map.items():
            if p2 in cl:
                if pd.api.types.is_numeric_dtype(df[c]) or pd.to_numeric(df[c], errors="coerce").notna().sum() > len(df) * 0.8:
                    return c

    # second pass: any column containing "wind"
    for c, cl in lower_map.items():
        if "wind" in cl:
            if pd.api.types.is_numeric_dtype(df[c]) or pd.to_numeric(df[c], errors="coerce").notna().sum() > len(df) * 0.8:
                return c

    # fallback: pick numeric column with largest non-null count, excluding obvious non-speed columns
    bad_keywords = ["power", "energy", "rotor", "rpm", "temp", "pitch", "yaw", "status", "error"]
    candidate_cols = []
    for c in cols:
        cl = str(c).lower()
        if any(b in cl for b in bad_keywords):
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        valid = s.notna().sum()
        if valid > len(df) * 0.8:
            candidate_cols.append((c, valid))

    if not candidate_cols:
        raise ValueError("Cannot detect a valid wind speed column.")
    candidate_cols = sorted(candidate_cols, key=lambda x: -x[1])
    return candidate_cols[0][0]


# ---------- load ----------
df_raw = pd.read_csv(CSV_PATH, low_memory=False)

time_col = detect_time_col(df_raw)
wind_col = detect_windspeed_col(df_raw)

print(f"[INFO] Detected time column      : {time_col}")
print(f"[INFO] Detected wind speed column: {wind_col}")

df = df_raw[[time_col, wind_col]].copy()
df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df[wind_col] = pd.to_numeric(df[wind_col], errors="coerce")
df = df.dropna(subset=[time_col, wind_col]).sort_values(time_col).reset_index(drop=True)

# remove duplicates
df = df.drop_duplicates(subset=[time_col]).copy()

# set index
df = df.set_index(time_col)
df = df.rename(columns={wind_col: "wind_speed"})

# try infer sampling interval
dt_sec = df.index.to_series().diff().dt.total_seconds().dropna()
median_dt = dt_sec.median() if len(dt_sec) else np.nan
print(f"[INFO] Median sampling interval: {median_dt:.2f} seconds")

# If not exactly 1 Hz, resample to 1-second grid using interpolation
if np.isnan(median_dt) or abs(median_dt - 1.0) > 0.2:
    print("[INFO] Data are not exactly 1 Hz. Resampling to 1-second series...")
    df_1s = df.resample("1S").mean().interpolate(method="time").ffill().bfill()
else:
    df_1s = df.copy()

print(f"[INFO] 1-second data shape: {df_1s.shape}")
print(df_1s.head())


# =========================================================
# 2. Build standard labels from high-frequency wind data
# =========================================================
"""
Labeling logic:
- WMO-style gust metric: max 3-second rolling mean within trailing 10-min window
- TI: std / mean within trailing 10-min window
- Gust factor: gust_3s / mean within trailing 10-min window
- IEC-inspired proxy: use TI >= 0.16 as strong turbulence proxy, with mean wind speed >= 3 m/s
  NOTE: this is an engineering proxy, not exact IEC extreme operating gust load formula.
"""

WINDOW_10MIN = "10min"
ROLL_3S = 3

# 3-second rolling mean gust metric (WMO-style gust intensity proxy)
df_1s["wind_3s_mean"] = df_1s["wind_speed"].rolling(f"{ROLL_3S}s", min_periods=ROLL_3S).mean()

# trailing 10-min features
feat = pd.DataFrame(index=df_1s.index)
feat["u_mean_10m"] = df_1s["wind_speed"].rolling(WINDOW_10MIN, min_periods=300).mean()
feat["u_std_10m"] = df_1s["wind_speed"].rolling(WINDOW_10MIN, min_periods=300).std()
feat["u_max3s_10m"] = df_1s["wind_3s_mean"].rolling(WINDOW_10MIN, min_periods=300).max()

# derived metrics
eps = 1e-6
feat["TI_10m"] = feat["u_std_10m"] / (feat["u_mean_10m"].abs() + eps)
feat["GF_10m"] = feat["u_max3s_10m"] / (feat["u_mean_10m"].abs() + eps)
feat["gust_excess_10m"] = feat["u_max3s_10m"] - feat["u_mean_10m"]

# drop initial NaNs
feat = feat.dropna().copy()

# -------------------------
# Threshold strategy
# -------------------------
# You can later tune these based on sensitivity analysis.
# Recommended: keep a hybrid of physical-ish floor + data-adaptive quantile.

# WMO-style gust threshold:
# classify as gusty when 3-s gust excess is unusually large
wmo_excess_floor = 2.0   # m/s floor
wmo_excess_q = feat["gust_excess_10m"].quantile(0.90)
WMO_EXCESS_TH = max(wmo_excess_floor, float(wmo_excess_q))

# TI threshold:
# 0.16 is often used as a meaningful severe-turbulence proxy in wind engineering discussions
# but we keep a data-adaptive fallback too
ti_floor = 0.16
ti_q = feat["TI_10m"].quantile(0.90)
TI_TH = max(ti_floor, float(ti_q * 0.9))   # slightly conservative

# Gust factor threshold
gf_floor = 1.25
gf_q = feat["GF_10m"].quantile(0.90)
GF_TH = max(gf_floor, float(gf_q))

# IEC-inspired proxy:
# engineering proxy: turbulence severe if TI >= 0.16 and mean wind speed >= 3 m/s
IEC_TI_TH = 0.16
IEC_MEAN_WS_TH = 3.0

# -------------------------
# Binary labels
# -------------------------
feat["label_wmo"] = (feat["gust_excess_10m"] >= WMO_EXCESS_TH).astype(int)
feat["label_ti"] = (feat["TI_10m"] >= TI_TH).astype(int)
feat["label_gf"] = (feat["GF_10m"] >= GF_TH).astype(int)
feat["label_iec_proxy"] = ((feat["TI_10m"] >= IEC_TI_TH) & (feat["u_mean_10m"] >= IEC_MEAN_WS_TH)).astype(int)

# Consensus label:
# majority vote among 4 rule families
rule_cols = ["label_wmo", "label_ti", "label_gf", "label_iec_proxy"]
feat["label_consensus"] = (feat[rule_cols].sum(axis=1) >= 2).astype(int)

# For comparison with Bollinger strategy, downsample to 1-minute evaluation grid
labels_1min = feat.resample("1min").last().dropna().copy()

print("\n[INFO] Thresholds used:")
print(f"WMO_EXCESS_TH = {WMO_EXCESS_TH:.4f} m/s")
print(f"TI_TH         = {TI_TH:.4f}")
print(f"GF_TH         = {GF_TH:.4f}")
print(f"IEC_TI_TH     = {IEC_TI_TH:.4f}")
print(f"IEC_MEAN_WS_TH= {IEC_MEAN_WS_TH:.4f} m/s")

print("\n[INFO] Label prevalence (1 = gusty / high-risk regime):")
for c in rule_cols + ["label_consensus"]:
    print(f"{c:18s}: {labels_1min[c].mean():.4%}")

# save label table
labels_1min.to_csv(os.path.join(OUT_DIR, "labels_1min_with_features.csv"))


# =========================================================
# 3. Bollinger-band financial strategy on 1-minute series
# =========================================================
# 1-minute mean wind speed for strategy input
u_1min = df_1s["wind_speed"].resample("1min").mean().rename("wind_1min")
u_1min = u_1min.to_frame()

# align with labels
eval_df = labels_1min.join(u_1min, how="inner").dropna().copy()

# Bollinger params (same spirit as your code)
window = 500   # 500 minutes
k = 1.0
beta = 1.0     # volatility amplification threshold multiplier

eval_df["middle_band"] = eval_df["wind_1min"].rolling(window=window, min_periods=window).mean()
eval_df["rolling_std"] = eval_df["wind_1min"].rolling(window=window, min_periods=window).std()
eval_df["upper_band"] = eval_df["middle_band"] + k * eval_df["rolling_std"]
eval_df["lower_band"] = eval_df["middle_band"] - k * eval_df["rolling_std"]

# historical baseline volatility:
# use rolling std median over a calibration period / all available past windows
global_baseline_sigma = eval_df["rolling_std"].median()

# financial strategy binary label
cond_band_break = (
    (eval_df["wind_1min"] > eval_df["upper_band"]) |
    (eval_df["wind_1min"] < eval_df["lower_band"])
)

cond_volatility = (eval_df["rolling_std"] > beta * global_baseline_sigma)

eval_df["label_bollinger"] = ((cond_band_break) | (cond_volatility)).astype(int)

# remove warm-up region
eval_df = eval_df.dropna(subset=["middle_band", "rolling_std", "upper_band", "lower_band"]).copy()

# ground truth
y_true = eval_df["label_consensus"].astype(int).values
y_pred = eval_df["label_bollinger"].astype(int).values

# metrics
acc = accuracy_score(y_true, y_pred)
bacc = balanced_accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
kappa = cohen_kappa_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

print("\n[RESULT] Bollinger strategy vs consensus labels")
print(f"Accuracy          : {acc:.4f}")
print(f"Balanced Accuracy : {bacc:.4f}")
print(f"Precision         : {prec:.4f}")
print(f"Recall            : {rec:.4f}")
print(f"F1-score          : {f1:.4f}")
print(f"Cohen's Kappa     : {kappa:.4f}\n")
print(classification_report(
    y_true, y_pred,
    target_names=["Normal", "Gusty"],
    zero_division=0
))

# save metrics
metrics_df = pd.DataFrame({
    "metric": ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "cohen_kappa"],
    "value": [acc, bacc, prec, rec, f1, kappa]
})
metrics_df.to_csv(os.path.join(OUT_DIR, "bollinger_vs_labels_metrics.csv"), index=False)

pd.DataFrame(cm, index=["True_Normal", "True_Gusty"], columns=["Pred_Normal", "Pred_Gusty"]).to_csv(
    os.path.join(OUT_DIR, "confusion_matrix.csv")
)


# =========================================================
# 4. Pretty figures for paper
# =========================================================

# ---------- Figure 1: full timeline snapshot with labels + Bollinger ----------
def plot_timeline_snapshot(df_eval, start=None, end=None, filename="fig_timeline_snapshot.png"):
    # choose a representative segment if not provided:
    # take a period with many true gust labels
    tmp = df_eval.copy()
    if start is None or end is None:
        # rolling density of true gust labels
        density = tmp["label_consensus"].rolling(240, min_periods=60).mean()
        center = density.idxmax()
        start = center - pd.Timedelta(hours=12)
        end = center + pd.Timedelta(hours=12)

    seg = tmp.loc[start:end].copy()
    if seg.empty:
        print("[WARN] Empty segment for timeline plot.")
        return

    fig, ax = plt.subplots(figsize=(16, 6))

    ax.plot(seg.index, seg["wind_1min"], lw=1.2, label="1-min wind speed")
    ax.plot(seg.index, seg["middle_band"], lw=1.0, label="Middle band")
    ax.plot(seg.index, seg["upper_band"], lw=1.0, label="Upper band")
    ax.plot(seg.index, seg["lower_band"], lw=1.0, label="Lower band")
    ax.fill_between(seg.index, seg["lower_band"], seg["upper_band"], alpha=0.15, label="Bollinger envelope")

    # shade true gusty regime
    gust_mask = seg["label_consensus"] == 1
    ax.fill_between(seg.index, seg["wind_1min"].min() - 0.5, seg["wind_1min"].max() + 0.5,
                    where=gust_mask, alpha=0.12, step="pre", label="Consensus gust regime")

    # predicted gust markers
    pred_idx = seg.index[seg["label_bollinger"] == 1]
    ax.scatter(pred_idx, seg.loc[pred_idx, "wind_1min"], s=10, label="Bollinger positive", zorder=5)

    ax.set_title("Wind-speed timeline with consensus regime labels and Bollinger detections")
    ax.set_ylabel("Wind speed")
    ax.set_xlabel("Time")
    ax.legend(ncol=3, frameon=False)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(DateFormatter("%Y-%m-%d\n%H:%M"))

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# ---------- Figure 2: confusion matrix ----------
def plot_confusion_matrix(cm, filename="fig_confusion_matrix.png"):
    fig, ax = plt.subplots(figsize=(5.8, 5.2))
    im = ax.imshow(cm, cmap="Blues")

    labels = ["Normal", "Gusty"]
    ax.set_xticks(np.arange(2))
    ax.set_yticks(np.arange(2))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion matrix: Bollinger strategy vs consensus labels")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]}",
                    ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black",
                    fontsize=12, fontweight="bold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Count", rotation=270, labelpad=15)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# ---------- Figure 3: feature space (TI vs Gust Factor) ----------
def plot_feature_space(df_eval, filename="fig_feature_space.png", max_points=20000):
    feat_df = df_eval.copy()

    # subsample for aesthetics if needed
    if len(feat_df) > max_points:
        feat_df = feat_df.sample(max_points, random_state=42).sort_index()

    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    # correctness type
    conditions = []
    for _, r in feat_df.iterrows():
        t = r["label_consensus"]
        p = r["label_bollinger"]
        if t == 0 and p == 0:
            conditions.append("TN")
        elif t == 0 and p == 1:
            conditions.append("FP")
        elif t == 1 and p == 0:
            conditions.append("FN")
        else:
            conditions.append("TP")
    feat_df["case_type"] = conditions

    color_map = {
        "TN": "#4C78A8",
        "FP": "#F58518",
        "FN": "#E45756",
        "TP": "#54A24B"
    }

    for ctype in ["TN", "FP", "FN", "TP"]:
        part = feat_df[feat_df["case_type"] == ctype]
        if len(part) == 0:
            continue
        ax.scatter(
            part["TI_10m"], part["GF_10m"],
            s=14, alpha=0.45, c=color_map[ctype], label=ctype, edgecolors="none"
        )

    ax.axvline(TI_TH, ls="--", lw=1.2, color="black", alpha=0.8, label=f"TI threshold = {TI_TH:.2f}")
    ax.axhline(GF_TH, ls=":", lw=1.4, color="black", alpha=0.8, label=f"GF threshold = {GF_TH:.2f}")

    ax.set_xlabel("Turbulence intensity (10-min)")
    ax.set_ylabel("Gust factor (10-min)")
    ax.set_title("Feature-space view of consensus labels and Bollinger outcomes")
    ax.legend(frameon=False, ncol=2)
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# ---------- Figure 4: event-level zoomed examples ----------
def get_contiguous_segments(binary_series, min_len=5):
    """
    Return list of (start_idx, end_idx) for contiguous 1-regions.
    min_len in number of samples.
    """
    x = binary_series.astype(int).values
    idx = binary_series.index

    segments = []
    in_seg = False
    s = None
    for i, val in enumerate(x):
        if val == 1 and not in_seg:
            in_seg = True
            s = i
        elif val == 0 and in_seg:
            e = i - 1
            if e - s + 1 >= min_len:
                segments.append((idx[s], idx[e]))
            in_seg = False
    if in_seg:
        e = len(x) - 1
        if e - s + 1 >= min_len:
            segments.append((idx[s], idx[e]))
    return segments


def plot_event_examples(df_eval, filename="fig_event_examples.png", n_events=3):
    segments = get_contiguous_segments(df_eval["label_consensus"], min_len=10)
    if len(segments) == 0:
        print("[WARN] No contiguous gust events found for example plot.")
        return

    # rank by event intensity
    ranked = []
    for s, e in segments:
        part = df_eval.loc[s:e]
        score = part["gust_excess_10m"].mean() + part["GF_10m"].mean()
        ranked.append((s, e, score))
    ranked = sorted(ranked, key=lambda x: -x[2])[:n_events]

    fig, axes = plt.subplots(n_events, 1, figsize=(15, 4.2 * n_events), sharex=False)
    if n_events == 1:
        axes = [axes]

    for ax, (s, e, _) in zip(axes, ranked):
        pad = pd.Timedelta(hours=2)
        part = df_eval.loc[s - pad:e + pad].copy()
        if part.empty:
            continue

        ax.plot(part.index, part["wind_1min"], lw=1.1, label="1-min wind speed")
        ax.plot(part.index, part["upper_band"], lw=0.9, label="Upper band")
        ax.plot(part.index, part["lower_band"], lw=0.9, label="Lower band")
        ax.fill_between(part.index, part["lower_band"], part["upper_band"], alpha=0.15)

        # true event
        ax.axvspan(s, e, alpha=0.16, label="Consensus gust event")

        # predicted positives
        pred_idx = part.index[part["label_bollinger"] == 1]
        ax.scatter(pred_idx, part.loc[pred_idx, "wind_1min"], s=10, label="Bollinger positive", zorder=5)

        ax.set_title(f"Example gust event: {s} to {e}")
        ax.set_ylabel("Wind speed")
        ax.grid(alpha=0.25)
        ax.legend(frameon=False, ncol=4)
        ax.xaxis.set_major_formatter(DateFormatter("%m-%d\n%H:%M"))

    axes[-1].set_xlabel("Time")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# ---------- Figure 5: label agreement bars ----------
def plot_label_agreement(df_eval, filename="fig_label_agreement.png"):
    counts = {
        "WMO": df_eval["label_wmo"].mean(),
        "TI": df_eval["label_ti"].mean(),
        "GF": df_eval["label_gf"].mean(),
        "IEC-proxy": df_eval["label_iec_proxy"].mean(),
        "Consensus": df_eval["label_consensus"].mean(),
        "Bollinger": df_eval["label_bollinger"].mean(),
    }

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    keys = list(counts.keys())
    vals = [counts[k] * 100 for k in keys]

    bars = ax.bar(keys, vals, alpha=0.85)
    ax.set_ylabel("Positive-rate (%)")
    ax.set_title("Positive-rate comparison across labeling rules and Bollinger strategy")
    ax.grid(axis="y", alpha=0.25)

    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# =========================================================
# 5. Run all figure makers
# =========================================================
plot_timeline_snapshot(eval_df, filename="fig1_timeline_snapshot.png")
plot_confusion_matrix(cm, filename="fig2_confusion_matrix.png")
plot_feature_space(eval_df, filename="fig3_feature_space.png")
plot_event_examples(eval_df, filename="fig4_event_examples.png", n_events=3)
plot_label_agreement(eval_df, filename="fig5_label_agreement.png")

print(f"\n[INFO] All outputs saved to: {os.path.abspath(OUT_DIR)}")
print("[INFO] Main outputs:")
print(" - labels_1min_with_features.csv")
print(" - bollinger_vs_labels_metrics.csv")
print(" - confusion_matrix.csv")
print(" - fig1_timeline_snapshot.png")
print(" - fig2_confusion_matrix.png")
print(" - fig3_feature_space.png")
print(" - fig4_event_examples.png")
print(" - fig5_label_agreement.png")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    cohen_kappa_score
)

# 输出目录
OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# 0. 安全检查
# =========================
required_cols = ["wind_1min", "label_consensus"]
for c in required_cols:
    if c not in eval_df.columns:
        raise ValueError(f"eval_df 中缺少必要列: {c}")

base_df = eval_df[["wind_1min", "label_consensus"]].copy()
base_df = base_df.dropna().copy()

y_true = base_df["label_consensus"].astype(int).values

print(f"[INFO] Samples for parameter sweep: {len(base_df)}")
print(f"[INFO] Positive rate of consensus label: {base_df['label_consensus'].mean():.4%}")


# =========================
# 1. 单组参数评估函数
# =========================
def evaluate_bollinger_params(df, window=500, k=1.0, beta=1.0):
    tmp = df.copy()

    tmp["middle_band"] = tmp["wind_1min"].rolling(window=window, min_periods=window).mean()
    tmp["rolling_std"] = tmp["wind_1min"].rolling(window=window, min_periods=window).std()
    tmp["upper_band"] = tmp["middle_band"] + k * tmp["rolling_std"]
    tmp["lower_band"] = tmp["middle_band"] - k * tmp["rolling_std"]

    tmp = tmp.dropna(subset=["middle_band", "rolling_std", "upper_band", "lower_band"]).copy()

    if len(tmp) == 0:
        return None

    global_baseline_sigma = tmp["rolling_std"].median()

    cond_band_break = (
        (tmp["wind_1min"] > tmp["upper_band"]) |
        (tmp["wind_1min"] < tmp["lower_band"])
    )
    cond_volatility = (tmp["rolling_std"] > beta * global_baseline_sigma)

    tmp["label_bollinger"] = ((cond_band_break) | (cond_volatility)).astype(int)

    y_true_local = tmp["label_consensus"].astype(int).values
    y_pred_local = tmp["label_bollinger"].astype(int).values

    # 避免极端情况下 precision 等报错
    acc = accuracy_score(y_true_local, y_pred_local)
    bacc = balanced_accuracy_score(y_true_local, y_pred_local)
    prec = precision_score(y_true_local, y_pred_local, zero_division=0)
    rec = recall_score(y_true_local, y_pred_local, zero_division=0)
    f1 = f1_score(y_true_local, y_pred_local, zero_division=0)
    kappa = cohen_kappa_score(y_true_local, y_pred_local)
    pred_pos_rate = np.mean(y_pred_local)

    return {
        "window": window,
        "k": k,
        "beta": beta,
        "n_samples": len(tmp),
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "kappa": kappa,
        "pred_positive_rate": pred_pos_rate
    }


# =========================
# 2. 参数网格
# =========================
window_grid = [120, 240, 360, 500, 720]
k_grid = [1.0, 1.5, 2.0, 2.5]
beta_grid = [1.0, 1.2, 1.5, 2.0, 2.5]

results = []

for window in window_grid:
    for k in k_grid:
        for beta in beta_grid:
            res = evaluate_bollinger_params(base_df, window=window, k=k, beta=beta)
            if res is not None:
                results.append(res)
                print(
                    f"[DONE] window={window:>3d}, k={k:.1f}, beta={beta:.1f} | "
                    f"F1={res['f1']:.4f}, Kappa={res['kappa']:.4f}, "
                    f"Prec={res['precision']:.4f}, Rec={res['recall']:.4f}, "
                    f"PosRate={res['pred_positive_rate']:.4%}"
                )

results_df = pd.DataFrame(results).sort_values(
    by=["f1", "kappa", "balanced_accuracy"], ascending=False
).reset_index(drop=True)

results_df.to_csv(os.path.join(OUT_DIR, "bollinger_parameter_sweep.csv"), index=False)

print("\n[INFO] Top 15 parameter combinations:")
print(results_df.head(15))

best_row = results_df.iloc[0]
print("\n[INFO] Best setting by F1/Kappa/BAcc ranking:")
print(best_row)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def plot_heatmap_for_window(results_df, fixed_window, metric="f1", cmap="viridis", filename=None):
    sub = results_df[results_df["window"] == fixed_window].copy()
    if sub.empty:
        print(f"[WARN] No data for window={fixed_window}")
        return

    pivot = sub.pivot(index="beta", columns="k", values=metric)
    pivot = pivot.sort_index().sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(6.8, 5.5))
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap)

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels([f"{x:.1f}" for x in pivot.columns])
    ax.set_yticklabels([f"{x:.1f}" for x in pivot.index])

    ax.set_xlabel("Band width coefficient k")
    ax.set_ylabel("Volatility threshold multiplier β")
    ax.set_title(f"{metric.upper()} heatmap (window = {fixed_window} min)")

    # 在格子中写数值
    values = pivot.values
    vmax = np.nanmax(values)
    vmin = np.nanmin(values)
    thresh = (vmax + vmin) / 2

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            val = values[i, j]
            ax.text(
                j, i, f"{val:.3f}",
                ha="center", va="center",
                color="white" if val > thresh else "black",
                fontsize=10, fontweight="bold"
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel(metric.upper(), rotation=270, labelpad=15)

    plt.tight_layout()

    if filename is None:
        filename = f"heatmap_{metric}_window_{fixed_window}.png"

    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()


# 你可以画多个窗口
for w in [120, 240, 500]:
    plot_heatmap_for_window(results_df, fixed_window=w, metric="f1", cmap="YlGnBu",
                            filename=f"fig_heatmap_f1_window_{w}.png")
    plot_heatmap_for_window(results_df, fixed_window=w, metric="kappa", cmap="OrRd",
                            filename=f"fig_heatmap_kappa_window_{w}.png")
    plot_heatmap_for_window(results_df, fixed_window=w, metric="recall", cmap="PuBuGn",
                            filename=f"fig_heatmap_recall_window_{w}.png")

In [ ]:
import matplotlib.pyplot as plt
import os

OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# 按 window 分组画散点
fig, ax = plt.subplots(figsize=(7.2, 6.0))

windows_sorted = sorted(results_df["window"].unique())
markers = ["o", "s", "^", "D", "P", "X", "v"]

for i, w in enumerate(windows_sorted):
    sub = results_df[results_df["window"] == w].copy()
    ax.scatter(
        sub["recall"], sub["precision"],
        s=80 * (sub["f1"] / sub["f1"].max().clip(lower=1e-6)),
        alpha=0.65,
        marker=markers[i % len(markers)],
        label=f"window={w}"
    )

# 标出 top 5
top5 = results_df.head(5).copy()
for _, row in top5.iterrows():
    ax.annotate(
        f"w={int(row['window'])}, k={row['k']:.1f}, β={row['beta']:.1f}",
        (row["recall"], row["precision"]),
        textcoords="offset points", xytext=(6, 6),
        fontsize=9
    )

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall trade-off of Bollinger strategy")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_pr_tradeoff.png"), bbox_inches="tight")
plt.close()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# 如果你没装 seaborn，可以删掉这一段，下面我也给纯 matplotlib 版本
try:
    import seaborn as sns

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    sns.boxplot(data=results_df, x="window", y="f1", ax=axes[0])
    axes[0].set_title("F1 across different windows")
    axes[0].set_xlabel("window (min)")
    axes[0].set_ylabel("F1")

    sns.boxplot(data=results_df, x="k", y="f1", ax=axes[1])
    axes[1].set_title("F1 across different k")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("F1")

    sns.boxplot(data=results_df, x="beta", y="f1", ax=axes[2])
    axes[2].set_title("F1 across different β")
    axes[2].set_xlabel("β")
    axes[2].set_ylabel("F1")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_f1_boxplots.png"), bbox_inches="tight")
    plt.close()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

    sns.boxplot(data=results_df, x="window", y="kappa", ax=axes[0])
    axes[0].set_title("Kappa across different windows")
    axes[0].set_xlabel("window (min)")
    axes[0].set_ylabel("Kappa")

    sns.boxplot(data=results_df, x="k", y="kappa", ax=axes[1])
    axes[1].set_title("Kappa across different k")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Kappa")

    sns.boxplot(data=results_df, x="beta", y="kappa", ax=axes[2])
    axes[2].set_title("Kappa across different β")
    axes[2].set_xlabel("β")
    axes[2].set_ylabel("Kappa")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "fig_kappa_boxplots.png"), bbox_inches="tight")
    plt.close()

except ImportError:
    print("[INFO] seaborn not installed, skip boxplots.")

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter

OUT_DIR = "regime_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

best_row = results_df.iloc[0]
best_window = int(best_row["window"])
best_k = float(best_row["k"])
best_beta = float(best_row["beta"])

print(f"[INFO] Best params -> window={best_window}, k={best_k}, beta={best_beta}")

best_df = base_df.copy()
best_df["middle_band"] = best_df["wind_1min"].rolling(window=best_window, min_periods=best_window).mean()
best_df["rolling_std"] = best_df["wind_1min"].rolling(window=best_window, min_periods=best_window).std()
best_df["upper_band"] = best_df["middle_band"] + best_k * best_df["rolling_std"]
best_df["lower_band"] = best_df["middle_band"] - best_k * best_df["rolling_std"]

best_df = best_df.dropna().copy()
global_baseline_sigma = best_df["rolling_std"].median()

cond_band_break = (
    (best_df["wind_1min"] > best_df["upper_band"]) |
    (best_df["wind_1min"] < best_df["lower_band"])
)
cond_volatility = (best_df["rolling_std"] > best_beta * global_baseline_sigma)

best_df["label_bollinger_best"] = ((cond_band_break) | (cond_volatility)).astype(int)

# 自动找 gust 密集区
density = best_df["label_consensus"].rolling(240, min_periods=60).mean()
center = density.idxmax()
start = center - pd.Timedelta(hours=12)
end = center + pd.Timedelta(hours=12)
seg = best_df.loc[start:end].copy()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(seg.index, seg["wind_1min"], lw=1.2, label="1-min wind speed")
ax.plot(seg.index, seg["middle_band"], lw=1.0, label="Middle band")
ax.plot(seg.index, seg["upper_band"], lw=1.0, label="Upper band")
ax.plot(seg.index, seg["lower_band"], lw=1.0, label="Lower band")
ax.fill_between(seg.index, seg["lower_band"], seg["upper_band"], alpha=0.15, label="Bollinger envelope")

# 真值标签区域
gust_mask = seg["label_consensus"] == 1
ax.fill_between(
    seg.index,
    seg["wind_1min"].min() - 0.5,
    seg["wind_1min"].max() + 0.5,
    where=gust_mask,
    alpha=0.12,
    step="pre",
    label="Consensus gust regime"
)

# 预测点
pred_idx = seg.index[seg["label_bollinger_best"] == 1]
ax.scatter(pred_idx, seg.loc[pred_idx, "wind_1min"], s=10, label="Best Bollinger positive", zorder=5)

ax.set_title(
    f"Best Bollinger strategy vs consensus labels "
    f"(window={best_window}, k={best_k:.1f}, β={best_beta:.1f})"
)
ax.set_ylabel("Wind speed")
ax.set_xlabel("Time")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=3)
ax.xaxis.set_major_formatter(DateFormatter("%Y-%m-%d\n%H:%M"))

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_best_bollinger_timeline.png"), bbox_inches="tight")
plt.close()

In [ ]:
# -*- coding: utf-8 -*-
"""
整体跨尺度识别实验（简化版，不经过插值）
目标：
1. 用真实 1 Hz 数据构造“标准高频标签”
2. 用真实 1 Hz 数据运行 Bollinger 高频识别
3. 将两种高频标签都投票/统计回 10 分钟标签
4. 比较 10 分钟标签的一致性
5. 输出 Accuracy / Precision / Recall / F1 / Kappa
6. 生成论文可展示图片
"""

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report
)

# =========================================================
# 0. 基础设置
# =========================================================
OUT_DIR = "cross_scale_vote_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams["figure.dpi"] = 160
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# =========================================================
# 1. 数据准备
# =========================================================
# 你如果已经有 df_1s，就直接用
# 要求：
# df_1s.index = DatetimeIndex
# df_1s["wind_speed"] = 1Hz 风速

# 例如：
# df_1s = df.copy()
# df_1s.index = pd.to_datetime(df_1s.index)
# df_1s = df_1s.sort_index()

if "df_1s" not in globals():
    raise ValueError("请先准备好 df_1s，且包含 wind_speed 列。")

df_1s = df_1s.copy()
df_1s = df_1s.sort_index()
df_1s = df_1s[["wind_speed"]].dropna().copy()

print(f"[INFO] 1Hz sample size: {len(df_1s)}")
print(df_1s.head())

# =========================================================
# 2. 真实 1Hz 数据上的“标准高频标签”
# =========================================================
"""
标准标签：
- WMO-style gust excess
- TI
- Gust Factor
- IEC-inspired turbulence proxy
最后做 consensus
全部在真实 1Hz 数据上构造
"""

WINDOW_10MIN = "10min"
ROLL_3S = 3
eps = 1e-6

# 3秒滑动平均
df_1s["wind_3s_mean"] = df_1s["wind_speed"].rolling(f"{ROLL_3S}s", min_periods=ROLL_3S).mean()

# 高频特征
feat = pd.DataFrame(index=df_1s.index)
feat["u_mean_10m"] = df_1s["wind_speed"].rolling(WINDOW_10MIN, min_periods=300).mean()
feat["u_std_10m"] = df_1s["wind_speed"].rolling(WINDOW_10MIN, min_periods=300).std()
feat["u_max3s_10m"] = df_1s["wind_3s_mean"].rolling(WINDOW_10MIN, min_periods=300).max()

feat["TI_10m"] = feat["u_std_10m"] / (feat["u_mean_10m"].abs() + eps)
feat["GF_10m"] = feat["u_max3s_10m"] / (feat["u_mean_10m"].abs() + eps)
feat["gust_excess_10m"] = feat["u_max3s_10m"] - feat["u_mean_10m"]

feat = feat.dropna().copy()

# 阈值
wmo_excess_floor = 2.0
wmo_excess_q = feat["gust_excess_10m"].quantile(0.90)
WMO_EXCESS_TH = max(wmo_excess_floor, float(wmo_excess_q))

ti_floor = 0.16
ti_q = feat["TI_10m"].quantile(0.90)
TI_TH = max(ti_floor, float(ti_q * 0.9))

gf_floor = 1.25
gf_q = feat["GF_10m"].quantile(0.90)
GF_TH = max(gf_floor, float(gf_q))

IEC_TI_TH = 0.16
IEC_MEAN_WS_TH = 3.0

feat["label_wmo"] = (feat["gust_excess_10m"] >= WMO_EXCESS_TH).astype(int)
feat["label_ti"] = (feat["TI_10m"] >= TI_TH).astype(int)
feat["label_gf"] = (feat["GF_10m"] >= GF_TH).astype(int)
feat["label_iec_proxy"] = ((feat["TI_10m"] >= IEC_TI_TH) & (feat["u_mean_10m"] >= IEC_MEAN_WS_TH)).astype(int)

rule_cols = ["label_wmo", "label_ti", "label_gf", "label_iec_proxy"]
feat["label_consensus_hf"] = (feat[rule_cols].sum(axis=1) >= 2).astype(int)

print("\n[INFO] Standard-label thresholds:")
print(f"WMO_EXCESS_TH = {WMO_EXCESS_TH:.4f}")
print(f"TI_TH         = {TI_TH:.4f}")
print(f"GF_TH         = {GF_TH:.4f}")
print(f"IEC_TI_TH     = {IEC_TI_TH:.4f}")
print(f"IEC_MEAN_WS_TH= {IEC_MEAN_WS_TH:.4f}")

print("\n[INFO] High-frequency positive rates:")
for c in rule_cols + ["label_consensus_hf"]:
    print(f"{c:20s}: {feat[c].mean():.4%}")

# =========================================================
# 3. 真实 1Hz 数据上的 Bollinger 高频标签
# =========================================================
"""
这里不经过插值，直接在真实 1Hz 上运行 Bollinger。
为了避免 1Hz 太噪声，仍然建议先转成 1-minute 风速序列，
然后做 Bollinger，再扩展回对应分钟内的高频标签。
这更接近你原文“高频统计识别再回投票”的思想。
"""

# 1分钟平均风速
u_1min = df_1s["wind_speed"].resample("1min").mean().to_frame("wind_1min")

# 与标准特征对齐到 1分钟
feat_1min = feat.resample("1min").last().dropna().copy()
eval_1min = feat_1min.join(u_1min, how="inner").dropna().copy()

# 你参数扫描得到的最优参数
window = 120   # min
k = 2.5
beta = 1.5

eval_1min["middle_band"] = eval_1min["wind_1min"].rolling(window=window, min_periods=window).mean()
eval_1min["rolling_std"] = eval_1min["wind_1min"].rolling(window=window, min_periods=window).std()
eval_1min["upper_band"] = eval_1min["middle_band"] + k * eval_1min["rolling_std"]
eval_1min["lower_band"] = eval_1min["middle_band"] - k * eval_1min["rolling_std"]

eval_1min = eval_1min.dropna(subset=["middle_band", "rolling_std", "upper_band", "lower_band"]).copy()
global_baseline_sigma = eval_1min["rolling_std"].median()

cond_band_break = (
    (eval_1min["wind_1min"] > eval_1min["upper_band"]) |
    (eval_1min["wind_1min"] < eval_1min["lower_band"])
)
cond_volatility = (eval_1min["rolling_std"] > beta * global_baseline_sigma)

eval_1min["label_bollinger_1min"] = ((cond_band_break) | (cond_volatility)).astype(int)

# =========================================================
# 4. 两种方法都投票回 10 分钟标签
# =========================================================
"""
标准方法：
- 先有高频 consensus label
- 再按 10 分钟窗口投票/占比阈值得到 10 分钟真值标签

Bollinger 方法：
- 先得到 1-minute label
- 再按 10 分钟窗口投票得到 10 分钟预测标签
"""

# ---- 标准真值：从高频标签聚合到 10分钟 ----
# 你文章里是多数投票，这里可以设分钟/秒级占比阈值
# 高频真值采用：10分钟窗口内，gusty占比 >= 50% 判为 gusty
truth_10min = feat[["label_consensus_hf"]].copy()
truth_10min = truth_10min.resample("10min").mean().rename(columns={"label_consensus_hf": "truth_ratio"})
truth_10min["label_truth_10min"] = (truth_10min["truth_ratio"] >= 0.5).astype(int)

# 可选：如果你想更宽松，可改 0.3 或 0.4
TRUTH_RATIO_TH = 0.5
truth_10min["label_truth_10min"] = (truth_10min["truth_ratio"] >= TRUTH_RATIO_TH).astype(int)

# ---- Bollinger预测：从1分钟标签聚合到10分钟 ----
pred_10min = eval_1min[["label_bollinger_1min"]].copy()
pred_10min = pred_10min.resample("10min").mean().rename(columns={"label_bollinger_1min": "pred_ratio"})
pred_10min["label_pred_10min"] = (pred_10min["pred_ratio"] >= 0.5).astype(int)

PRED_RATIO_TH = 0.5
pred_10min["label_pred_10min"] = (pred_10min["pred_ratio"] >= PRED_RATIO_TH).astype(int)

# ---- 对齐 ----
compare_10min = truth_10min.join(pred_10min, how="inner").dropna().copy()

# 附加 10分钟风速均值，方便画图
u_10min = df_1s["wind_speed"].resample("10min").mean().rename("wind_10min")
compare_10min = compare_10min.join(u_10min, how="left")

print(f"\n[INFO] 10-min aligned sample size: {len(compare_10min)}")
print(f"[INFO] 10-min truth positive rate: {compare_10min['label_truth_10min'].mean():.4%}")
print(f"[INFO] 10-min pred  positive rate: {compare_10min['label_pred_10min'].mean():.4%}")

# =========================================================
# 5. 计算 10 分钟标签一致性指标
# =========================================================
y_true_10 = compare_10min["label_truth_10min"].astype(int).values
y_pred_10 = compare_10min["label_pred_10min"].astype(int).values

acc = accuracy_score(y_true_10, y_pred_10)
bacc = balanced_accuracy_score(y_true_10, y_pred_10)
prec = precision_score(y_true_10, y_pred_10, zero_division=0)
rec = recall_score(y_true_10, y_pred_10, zero_division=0)
f1 = f1_score(y_true_10, y_pred_10, zero_division=0)
kappa = cohen_kappa_score(y_true_10, y_pred_10)

cm = confusion_matrix(y_true_10, y_pred_10)

print("\n[RESULT] 10-min label agreement: standard voting vs Bollinger voting")
print(f"Accuracy          : {acc:.4f}")
print(f"Balanced Accuracy : {bacc:.4f}")
print(f"Precision         : {prec:.4f}")
print(f"Recall            : {rec:.4f}")
print(f"F1-score          : {f1:.4f}")
print(f"Cohen's Kappa     : {kappa:.4f}\n")

print(classification_report(
    y_true_10, y_pred_10,
    target_names=["Normal_10min", "Gusty_10min"],
    zero_division=0
))

# 保存结果
metrics_df = pd.DataFrame({
    "metric": ["accuracy", "balanced_accuracy", "precision", "recall", "f1", "cohen_kappa"],
    "value": [acc, bacc, prec, rec, f1, kappa]
})
metrics_df.to_csv(os.path.join(OUT_DIR, "ten_min_vote_metrics.csv"), index=False)

pd.DataFrame(cm, index=["True_Normal", "True_Gusty"], columns=["Pred_Normal", "Pred_Gusty"]).to_csv(
    os.path.join(OUT_DIR, "ten_min_vote_confusion_matrix.csv")
)

compare_10min.to_csv(os.path.join(OUT_DIR, "ten_min_vote_comparison.csv"))

# =========================================================
# 6. 可视化 1：10分钟标签时间轴
# =========================================================
def plot_10min_timeline(compare_df, filename="fig_10min_timeline.png"):
    # 自动找 gust 最密集区域
    density = compare_df["label_truth_10min"].rolling(72, min_periods=12).mean()  # 12小时窗口
    center = density.idxmax()
    start = center - pd.Timedelta(hours=24)
    end = center + pd.Timedelta(hours=24)

    seg = compare_df.loc[start:end].copy()
    if seg.empty:
        seg = compare_df.iloc[:min(500, len(compare_df))].copy()

    fig, ax1 = plt.subplots(figsize=(16, 6))

    # 风速曲线
    ax1.plot(seg.index, seg["wind_10min"], lw=1.4, label="10-min mean wind speed")
    ax1.set_ylabel("Wind speed")
    ax1.set_xlabel("Time")
    ax1.grid(alpha=0.25)

    # 真值区域
    truth_mask = seg["label_truth_10min"] == 1
    pred_mask = seg["label_pred_10min"] == 1

    ymin = seg["wind_10min"].min() - 0.5
    ymax = seg["wind_10min"].max() + 0.5

    ax1.fill_between(seg.index, ymin, ymax, where=truth_mask, alpha=0.14, step="pre",
                     label="10-min truth regime")
    ax1.fill_between(seg.index, ymin, ymax, where=pred_mask, alpha=0.10, step="pre",
                     label="10-min Bollinger-voted regime")

    ax1.set_title("10-min regime labels obtained by voting: standard rules vs Bollinger strategy")
    ax1.legend(frameon=False, ncol=3)
    ax1.xaxis.set_major_formatter(DateFormatter("%Y-%m-%d\n%H:%M"))

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()

# =========================================================
# 7. 可视化 2：混淆矩阵
# =========================================================
def plot_confusion_matrix(cm, filename="fig_10min_confusion_matrix.png"):
    fig, ax = plt.subplots(figsize=(5.6, 5.0))
    im = ax.imshow(cm, cmap="Blues")

    labels = ["Normal", "Gusty"]
    ax.set_xticks(np.arange(2))
    ax.set_yticks(np.arange(2))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted 10-min label")
    ax.set_ylabel("True 10-min label")
    ax.set_title("Confusion matrix of 10-min voted labels")

    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=12, fontweight="bold"
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Count", rotation=270, labelpad=15)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()

# =========================================================
# 8. 可视化 3：truth_ratio vs pred_ratio 散点
# =========================================================
def plot_ratio_scatter(compare_df, filename="fig_ratio_scatter.png", max_points=12000):
    dfp = compare_df.copy()
    if len(dfp) > max_points:
        dfp = dfp.sample(max_points, random_state=42)

    fig, ax = plt.subplots(figsize=(6.8, 6.0))
    ax.scatter(dfp["truth_ratio"], dfp["pred_ratio"], s=18, alpha=0.35)

    lim = [0, 1]
    ax.plot(lim, lim, "--", lw=1.2, color="black", label="y = x")
    ax.axvline(TRUTH_RATIO_TH, ls="--", lw=1.2, label=f"truth threshold = {TRUTH_RATIO_TH:.1f}")
    ax.axhline(PRED_RATIO_TH, ls=":", lw=1.2, label=f"pred threshold = {PRED_RATIO_TH:.1f}")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Truth gust ratio within 10-min window")
    ax.set_ylabel("Predicted gust ratio within 10-min window")
    ax.set_title("Voting-ratio comparison at the 10-min scale")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()

# =========================================================
# 9. 可视化 4：四类错误案例分布
# =========================================================
def plot_case_distribution(compare_df, filename="fig_case_distribution.png"):
    dfp = compare_df.copy()
    case_type = np.where(
        (dfp["label_truth_10min"] == 0) & (dfp["label_pred_10min"] == 0), "TN",
        np.where(
            (dfp["label_truth_10min"] == 0) & (dfp["label_pred_10min"] == 1), "FP",
            np.where(
                (dfp["label_truth_10min"] == 1) & (dfp["label_pred_10min"] == 0), "FN", "TP"
            )
        )
    )
    dfp["case_type"] = case_type

    counts = dfp["case_type"].value_counts().reindex(["TN", "FP", "FN", "TP"]).fillna(0)

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    bars = ax.bar(counts.index, counts.values, alpha=0.85)
    ax.set_ylabel("Count")
    ax.set_title("Case distribution of 10-min voted labels")
    ax.grid(axis="y", alpha=0.25)

    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width()/2, v, f"{int(v)}", ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), bbox_inches="tight")
    plt.close()

# 跑图
plot_10min_timeline(compare_10min)
plot_confusion_matrix(cm)
plot_ratio_scatter(compare_10min)
plot_case_distribution(compare_10min)

print(f"\n[INFO] All outputs saved to: {os.path.abspath(OUT_DIR)}")
print("[INFO] Main outputs:")
print(" - ten_min_vote_metrics.csv")
print(" - ten_min_vote_confusion_matrix.csv")
print(" - ten_min_vote_comparison.csv")
print(" - fig_10min_timeline.png")
print(" - fig_10min_confusion_matrix.png")
print(" - fig_ratio_scatter.png")
print(" - fig_case_distribution.png")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import confusion_matrix

# =========================================================
# Configuration
# =========================================================
PARAM_SWEEP_CSV = os.environ.get("PARAM_SWEEP_CSV", "regime_eval_outputs/bollinger_parameter_sweep.csv")
TEN_MIN_COMPARE_CSV = os.environ.get("TEN_MIN_COMPARE_CSV", "cross_scale_vote_eval_outputs/ten_min_vote_comparison.csv")
OUT_DIR = os.environ.get("OUT_DIR", ".")
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================
# Unified paper-style plotting configuration
# =========================================================
plt.rcParams["figure.dpi"] = 180
plt.rcParams["savefig.dpi"] = 400
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.linewidth"] = 0.6
plt.rcParams["grid.alpha"] = 0.20
plt.rcParams["lines.linewidth"] = 1.5

COLORS = {
    "navy": "#1F3B73",
    "blue": "#2F6DB3",
    "light_blue": "#BFD7EA",
    "red": "#C44536",
    "orange": "#D98E04",
    "gold": "#E0B43B",
    "green": "#4E8D7C",
    "gray": "#6E6E6E",
    "light_gray": "#D9D9D9",
    "dark": "#2B2B2B",
    "white": "#FFFFFF"
}

kappa_cmap = LinearSegmentedColormap.from_list(
    "paper_orange_red",
    ["#FFF7EC", "#FDD49E", "#FC8D59", "#D7301F", "#7F0000"]
)

cm_cmap = LinearSegmentedColormap.from_list(
    "paper_blue",
    ["#F7FBFF", "#DEEBF7", "#9ECAE1", "#3182BD", "#08519C"]
)


def _save_close(fig, path):
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def _paper_grid(ax):
    ax.grid(True, axis="both", linestyle="--", alpha=0.18)


# =========================================================
# Robust CSV loading helpers
# =========================================================
def load_param_sweep(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Parameter sweep CSV not found: {path}")
    df = pd.read_csv(path)
    required = {"window", "k", "beta", "precision", "recall", "f1", "kappa", "balanced_accuracy"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in parameter sweep CSV: {missing}")
    return df


def load_ten_min_compare(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"10-min comparison CSV not found: {path}")
    # try common layouts
    df = pd.read_csv(path)
    # parse first datetime-like column as index
    datetime_col = None
    for c in df.columns:
        if str(c).lower() in ["datetime", "time", "timestamp", "date"] or "unnamed" in str(c).lower():
            try:
                parsed = pd.to_datetime(df[c], errors="coerce")
                if parsed.notna().mean() > 0.8:
                    datetime_col = c
                    df[c] = parsed
                    break
            except Exception:
                pass
    if datetime_col is not None:
        df = df.set_index(datetime_col)

    required = {"truth_ratio", "label_truth_10min", "pred_ratio", "label_pred_10min"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in 10-min comparison CSV: {missing}")

    # wind_10min is optional but strongly preferred
    if "wind_10min" not in df.columns:
        df["wind_10min"] = np.nan
    return df


# =========================================================
# Figure generators
# =========================================================
def plot_kappa_heatmap_paper(results_df, fixed_window=120, save_path="AGruPn_fig-param-kappa.png"):
    sub = results_df[results_df["window"] == fixed_window].copy()
    if sub.empty:
        raise ValueError(f"results_df has no rows with window={fixed_window}")

    pivot = sub.pivot(index="beta", columns="k", values="kappa")
    pivot = pivot.sort_index().sort_index(axis=1)

    fig, ax = plt.subplots(figsize=(6.8, 5.6))
    im = ax.imshow(pivot.values, aspect="auto", cmap=kappa_cmap)

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels([f"{x:.1f}" for x in pivot.columns])
    ax.set_yticklabels([f"{x:.1f}" for x in pivot.index])

    ax.set_xlabel("Band width coefficient $k$")
    ax.set_ylabel("Volatility threshold multiplier $\\beta$")
    ax.set_title(f"Parameter sensitivity of Cohen's kappa ($L={fixed_window}$ min)", pad=10)

    vals = pivot.values
    thresh = (np.nanmax(vals) + np.nanmin(vals)) / 2
    for i in range(vals.shape[0]):
        for j in range(vals.shape[1]):
            v = vals[i, j]
            ax.text(
                j, i, f"{v:.3f}",
                ha="center", va="center",
                color="white" if v > thresh else COLORS["dark"],
                fontsize=10, fontweight="semibold"
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Kappa", rotation=270, labelpad=15)

    _save_close(fig, os.path.join(OUT_DIR, save_path))


def plot_pr_tradeoff_paper(results_df, save_path="AGruPn_fig-param-pr.png"):
    fig, ax = plt.subplots(figsize=(7.2, 6.0))

    windows_sorted = sorted(results_df["window"].unique())
    marker_map = {120: "o", 240: "s", 360: "^", 500: "D", 720: "P"}
    color_map = {
        120: COLORS["navy"],
        240: COLORS["blue"],
        360: COLORS["green"],
        500: COLORS["orange"],
        720: COLORS["red"]
    }

    for w in windows_sorted:
        sub = results_df[results_df["window"] == w].copy()
        size_scale = sub["f1"] / max(sub["f1"].max(), 1e-6)
        ax.scatter(
            sub["recall"], sub["precision"],
            s=45 + 110 * size_scale,
            alpha=0.72,
            marker=marker_map.get(w, "o"),
            color=color_map.get(w, COLORS["gray"]),
            edgecolors="white",
            linewidths=0.7,
            label=f"$L={w}$"
        )

    top5 = results_df.sort_values(by=["f1", "kappa", "balanced_accuracy"], ascending=False).head(5)
    for _, row in top5.iterrows():
        ax.annotate(
            f"L={int(row['window'])}, k={row['k']:.1f}, β={row['beta']:.1f}",
            (row["recall"], row["precision"]),
            textcoords="offset points",
            xytext=(6, 6),
            fontsize=8.8,
            color=COLORS["dark"]
        )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision–recall trade-off under different parameter settings", pad=10)
    ax.set_xlim(0.15, min(1.0, results_df["recall"].max() + 0.05))
    ax.set_ylim(0.15, min(1.0, results_df["precision"].max() + 0.08))
    _paper_grid(ax)
    ax.legend(frameon=False, ncol=2)

    _save_close(fig, os.path.join(OUT_DIR, save_path))


def plot_10min_timeline_paper(compare_10min, save_path="AGruPn_fig-10min-timeline.png"):
    density = compare_10min["label_truth_10min"].rolling(72, min_periods=12).mean()
    center = density.idxmax()
    start = center - pd.Timedelta(hours=12)
    end = center + pd.Timedelta(hours=12)

    seg = compare_10min.loc[start:end].copy()
    if seg.empty:
        seg = compare_10min.iloc[:min(300, len(compare_10min))].copy()

    # 关键修改：正方形画布
    fig, ax = plt.subplots(figsize=(5.4, 5.4))

    ax.plot(
        seg.index, seg["wind_10min"],
        color=COLORS["navy"], lw=1.4, label="10-min mean wind speed"
    )

    truth_mask = seg["label_truth_10min"] == 1
    pred_mask = seg["label_pred_10min"] == 1

    ymin = seg["wind_10min"].min() - 0.5
    ymax = seg["wind_10min"].max() + 0.5

    ax.fill_between(
        seg.index, ymin, ymax,
        where=truth_mask,
        alpha=0.18, step="pre",
        color=COLORS["gold"],
        label="Rule-based label"
    )
    ax.fill_between(
        seg.index, ymin, ymax,
        where=pred_mask,
        alpha=0.10, step="pre",
        color=COLORS["red"],
        label="Bollinger-voted label"
    )

    ax.set_ylabel("Wind speed")
    ax.set_xlabel("Time")
    ax.grid(True, linestyle="--", alpha=0.18)

    # 不要太密的时间刻度
    locator_idx = np.linspace(0, len(seg.index) - 1, 4, dtype=int)
    tick_positions = seg.index[locator_idx]
    ax.set_xticks(tick_positions)
    ax.xaxis.set_major_formatter(DateFormatter("%m-%d\n%H:%M"))

    ax.legend(
        frameon=False,
        loc="upper left",
        fontsize=8,
        handlelength=1.6
    )

    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", facecolor="white")
    plt.close(fig)

def plot_confusion_matrix_paper(cm, save_path="AGruPn_fig-10min-cm.png"):
    fig, ax = plt.subplots(figsize=(5.6, 5.0))
    im = ax.imshow(cm, cmap=cm_cmap)

    labels = ["Normal", "Gusty"]
    ax.set_xticks(np.arange(2))
    ax.set_yticks(np.arange(2))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Confusion matrix of 10-min condition labels", pad=10)

    thresh = cm.max() / 2
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else COLORS["dark"],
                    fontsize=12, fontweight="semibold")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_ylabel("Count", rotation=270, labelpad=15)

    _save_close(fig, os.path.join(OUT_DIR, save_path))


def plot_ratio_scatter_paper(compare_10min, truth_th=0.5, pred_th=0.5, save_path="AGruPn_fig-10min-ratio.png"):
    dfp = compare_10min.copy()
    if len(dfp) > 12000:
        dfp = dfp.sample(12000, random_state=42)

    fig, ax = plt.subplots(figsize=(6.8, 6.0))
    ax.scatter(dfp["truth_ratio"], dfp["pred_ratio"], s=18, alpha=0.35,
               color=COLORS["blue"], edgecolors="none")

    lim = [0, 1]
    ax.plot(lim, lim, "--", lw=1.2, color=COLORS["dark"], label="$y=x$")
    ax.axvline(truth_th, ls="--", lw=1.2, color=COLORS["orange"], label=f"Truth threshold = {truth_th:.1f}")
    ax.axhline(pred_th, ls=":", lw=1.4, color=COLORS["red"], label=f"Prediction threshold = {pred_th:.1f}")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("True gust ratio within a 10-min window")
    ax.set_ylabel("Predicted gust ratio within a 10-min window")
    ax.set_title("Voting-ratio agreement at the 10-min scale", pad=10)
    _paper_grid(ax)
    ax.legend(frameon=False, loc="upper left")

    _save_close(fig, os.path.join(OUT_DIR, save_path))


def plot_case_distribution_paper(compare_10min, save_path="AGruPn_fig-10min-case.png"):
    dfp = compare_10min.copy()
    case_type = np.where(
        (dfp["label_truth_10min"] == 0) & (dfp["label_pred_10min"] == 0), "TN",
        np.where((dfp["label_truth_10min"] == 0) & (dfp["label_pred_10min"] == 1), "FP",
                 np.where((dfp["label_truth_10min"] == 1) & (dfp["label_pred_10min"] == 0), "FN", "TP"))
    )
    dfp["case_type"] = case_type
    counts = dfp["case_type"].value_counts().reindex(["TN", "FP", "FN", "TP"]).fillna(0)

    bar_colors = [COLORS["navy"], COLORS["orange"], COLORS["red"], COLORS["green"]]
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    bars = ax.bar(counts.index, counts.values, alpha=0.90, color=bar_colors,
                  edgecolor="white", linewidth=0.8)

    ax.set_ylabel("Count")
    ax.set_title("Distribution of case types for 10-min condition labels", pad=10)
    ax.grid(axis="y", linestyle="--", alpha=0.18)

    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{int(v)}",
                ha="center", va="bottom", fontsize=10, color=COLORS["dark"])

    _save_close(fig, os.path.join(OUT_DIR, save_path))


# =========================================================
# Main
# =========================================================
def main():
    results_df = load_param_sweep(PARAM_SWEEP_CSV)
    compare_10min = load_ten_min_compare(TEN_MIN_COMPARE_CSV)
    cm = confusion_matrix(compare_10min["label_truth_10min"].astype(int), compare_10min["label_pred_10min"].astype(int))

    plot_kappa_heatmap_paper(results_df, fixed_window=120, save_path="AGruPn_fig-param-kappa.png")
    plot_pr_tradeoff_paper(results_df, save_path="AGruPn_fig-param-pr.png")
    plot_10min_timeline_paper(compare_10min, save_path="AGruPn_fig-10min-timeline.png")
    plot_confusion_matrix_paper(cm, save_path="AGruPn_fig-10min-cm.png")
    plot_ratio_scatter_paper(compare_10min, truth_th=0.5, pred_th=0.5, save_path="AGruPn_fig-10min-ratio.png")
    plot_case_distribution_paper(compare_10min, save_path="AGruPn_fig-10min-case.png")

    print("[INFO] Final paper-style figures generated:")
    print(" - AGruPn_fig-param-kappa.png")
    print(" - AGruPn_fig-param-pr.png")
    print(" - AGruPn_fig-10min-timeline.png")
    print(" - AGruPn_fig-10min-cm.png")
    print(" - AGruPn_fig-10min-ratio.png")
    print(" - AGruPn_fig-10min-case.png")
    print(f"[INFO] Output directory: {os.path.abspath(OUT_DIR)}")


if __name__ == "__main__":
    main()
